# T07 - Evaluation: so sánh mô hình trên tập test
Tính **R², MAE, RMSE** trên tập test cho từng mô hình rồi so sánh.

- **R²:** mô hình giải thích được bao nhiêu phần biến động của giá (1 là hoàn hảo).
- **MAE:** sai số trung bình, tính bằng triệu đồng.
- **RMSE:** giống MAE nhưng phạt nặng các lần sai lớn, nên luôn lớn hơn hoặc bằng MAE.

Mô hình lấy từ T05 (`linear_regressor.pkl`) và T06 (`rf_baseline.pkl`, `rf_tuned.pkl`, `xgb_baseline.pkl`, `xgb_tuned.pkl`).
Cài một lần: `pip install pandas numpy scikit-learn xgboost joblib matplotlib ipykernel`

In [ ]:
import os
from pathlib import Path
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

TARGET = "price_million"

# Gốc repo: thư mục cha gần nhất có .git hoặc README.md
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / ".git").exists() or (p / "README.md").exists()), Path.cwd())

def find(name):
    """Tìm file theo tên trong repo; nếu có nhiều bản thì lấy bản mới nhất."""
    hits = []
    for folder, dirs, files in os.walk(ROOT):
        dirs[:] = [d for d in dirs if d not in {".venv", ".git", "__pycache__"}]
        if name in files:
            hits.append(Path(folder) / name)
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

print("Gốc repo:", ROOT)

## 1. Nạp tập test
Mô hình T05 chạy trên `data_test.csv` (đã One-Hot và Scale bởi T04). Mô hình T06 tự tiền xử lý nên chạy trên `test_set.csv` (cột gốc). Hai tập phải là **cùng các xe, cùng thứ tự** thì so sánh mới công bằng, nên kiểm tra giá của hai tập khớp nhau.

In [ ]:
path_lin = find("data_test.csv")
path_raw = find("test_set.csv")
print("T05 dùng:", path_lin)
print("T06 dùng:", path_raw)

test_lin = pd.read_csv(path_lin)
test_raw = pd.read_csv(path_raw)

same = len(test_lin) == len(test_raw) and np.allclose(test_lin[TARGET], test_raw[TARGET])
print("Hai tập test là cùng các xe, cùng thứ tự:", same, "| số xe:", len(test_raw))
assert same, "Hai tập test không khớp, không thể so sánh trực tiếp"

y_test = test_raw[TARGET]
X_lin = test_lin.drop(columns=TARGET)
X_raw = test_raw.drop(columns=TARGET)

## 2. Tính chỉ số cho từng mô hình

In [ ]:
MODELS = [
    ("Linear Regression (T05)", "linear_regressor.pkl", X_lin),
    ("Random Forest (baseline)", "rf_baseline.pkl", X_raw),
    ("Random Forest (tuned)", "rf_tuned.pkl", X_raw),
    ("XGBoost (baseline)", "xgb_baseline.pkl", X_raw),
    ("XGBoost (tuned)", "xgb_tuned.pkl", X_raw),
]
# Nếu T06 chưa chạy lại để xuất xgb_tuned.pkl, tạm dùng best_model.pkl (XGBoost tuned ở lần chạy trước)
if find("xgb_tuned.pkl") is None and find("best_model.pkl") is not None:
    MODELS[-1] = ("XGBoost (tuned)", "best_model.pkl", X_raw)

rows = []
for name, file, X in MODELS:
    path = find(file)
    if path is None:
        print(f"Bỏ qua {name}: chưa có file {file}")
        continue
    pred = joblib.load(path).predict(X)
    rows.append({
        "Mô hình": name,
        "R²": round(r2_score(y_test, pred), 3),
        "MAE (triệu)": round(mean_absolute_error(y_test, pred), 1),
        "RMSE (triệu)": round(float(np.sqrt(mean_squared_error(y_test, pred))), 1),
    })

results = pd.DataFrame(rows).sort_values("R²", ascending=False).reset_index(drop=True)
results

## 3. So sánh bằng biểu đồ

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
order = results["Mô hình"][::-1]
for a, col, color in zip(ax, ["R²", "MAE (triệu)", "RMSE (triệu)"], ["#4c78a8", "#59a14f", "#e15759"]):
    vals = results.set_index("Mô hình").loc[order, col]
    a.barh(vals.index, vals.values, color=color)
    a.set_title(col)
    for i, v in enumerate(vals.values):
        a.text(v, i, f" {v}", va="center")
    a.margins(x=0.15)
plt.tight_layout()
plt.show()

## 4. Kết luận
Điền sau khi xem bảng và biểu đồ:

- Mô hình tốt nhất (R² cao nhất, MAE và RMSE thấp nhất): ...
- So với Linear Regression (mốc cơ bản), mô hình tốt nhất cải thiện R² thêm ... và giảm MAE khoảng ... triệu.
- Tuning có cải thiện không? (so baseline với tuned của từng mô hình): ...
- RMSE lớn hơn nhiều so với MAE nghĩa là có một số xe bị dự đoán sai rất nhiều (thường là xe đắt): ...